Der Code unterhalb ist dafür da, damit Python die Funktionen nutzen kann, die wir hier später verwenden. Dieser Block muss unbedingt ausgeführt werden.

In [ ]:
import pandas as pd
import numpy as np

**Einlesen der CSV**

In [ ]:
csv_name = 'heart.csv'  # Datei muss in Colab hochgeladen werden oder Pfad anpassen
df = pd.read_csv(csv_name)
sample = df['heartrate'].copy()  # Spalte anpassen, falls notwendig
df['heartrate'] = df['heartrate'].astype(float)  # Convert heart rate to float
df['time'] = pd.to_datetime(df['time'])  # Convert time to datetime

# Prozentsatz der Daten
n = int(len(sample) * 0.03)

# Positionen
n_indices = np.random.choice(sample.index, size=n, replace=False)

# Setzen
sample.loc[n_indices] = np.random.choice([0, 700], size=n)


# Dataframe aus geladener CSV erstellen
df = pd.read_csv('heart.csv')

# Preview der ersten Reihen
df.head()

**Plot der eingelesenen Daten**

In [ ]:
if 'time' in df.columns:
    plt.figure(figsize=(8, 5))
    sns.lineplot(x=df['time'], y=sample, color='red')
    plt.xlabel('Zeit')
    plt.ylabel('Herzrate')
    plt.title('Liniendiagramm der Herzrate über die Zeit')
    plt.show()



---

**Empirische Regel**

* **68 % der Daten liegen innerhalb einer Standardabweichung vom Mittelwert**
* **95 % der Werte liegen innerhalb von zwei Standardabweichungen**
* **99,7 % der Werte liegen innerhalb von drei Standardabweichungen**

In [ ]:
# Mittelwert berechnen
mean_heart_rate = df['heartrate'].mean()
print(f"Mean heart rate: {mean_heart_rate}")

# Standardabweichung berechnen
std_heart_rate = df['heartrate'].std()
print(f"Standard deviation of heart rate: {std_heart_rate}")

**Berechnung wie viel Prozent der Daten innerhalb einer Standardabweichung vom Mittelwert liegen**

In [ ]:
# Bereich der ersten Standardabweichung
lower_bound = mean_heart_rate - std_heart_rate
upper_bound = mean_heart_rate + std_heart_rate

# Dataframe mit dem abgesteckten Bereich filtern
subset_df = df[(df['heartrate'] > lower_bound) & (df['heartrate'] < upper_bound)]

# Prozent berechnen
percentage = (len(subset_df) / len(df)) * 100

# Ausgabe
print(f"{percentage:.2f}% der Werte liegen zwischen {lower_bound:.2f} und {upper_bound:.2f}")


**Berechnung wie viel Prozent der Daten innerhalb von zwei Standardabweichungen vom Mittelwert liegen**

In [ ]:
# Bereich der zweiten Standardabweichung
lower_bound_2 = mean_heart_rate - 2 * std_heart_rate
upper_bound_2 = mean_heart_rate + 2 * std_heart_rate

# Dataframe mit dem abgesteckten Bereich filtern
subset_2 = df[(df['heartrate'] > lower_bound_2) & (df['heartrate'] < upper_bound_2)]

# Prozent berechnen
percentage_2 = (len(subset_2) / len(df)) * 100

# Ausgabe
print(f"{percentage_2:.2f}% der Werte liegen zwischen {lower_bound_2:.2f} und {upper_bound_2:.2f}")

**Berechnung wie viel Prozent der Daten innerhalb von drei Standardabweichungen vom Mittelwert liegen**

In [ ]:
# Bereich der dritten Standardabweichung
lower_bound_3 = mean_heart_rate - 3 * std_heart_rate
upper_bound_3 = mean_heart_rate + 3 * std_heart_rate

# Dataframe mit dem abgesteckten Bereich filtern
subset_3 = df[(df['heartrate'] > lower_bound_3) & (df['heartrate'] < upper_bound_3)]

# Prozent berechnen
percentage_3 = (len(subset_3) / len(df)) * 100

# Ausgabe
print(f"{percentage_3:.2f}% der Werte liegen zwischen {lower_bound_3:.2f} und {upper_bound_3:.2f}")

Alle Datenpunkte, die außerhalb des erwarteten Bereichs liegen (d.h. außerhalb von drei Standardabweichungen vom Mittelwert), werden als Ausreißer betrachtet.



---
**Korrektur der Daten mit Plot**
---
**Korrektur mit Top- und Bottom-Coding**

In [ ]:
#df = pd.DataFrame({'heart-rate': sample})
df['heart-rate_corrected_clip'] = df['heartrate'].clip(lower=lower_bound_3, upper=upper_bound_3)

# Plot
sample = df['heart-rate_corrected_clip'].copy()

if 'time' in df.columns:
    plt.figure(figsize=(8, 5))
    sns.lineplot(x=df['time'], y=sample, color='red')
    plt.xlabel('Zeit')
    plt.ylabel('Herzrate')
    plt.title('Liniendiagramm der Herzrate über die Zeit')
    plt.show()

**Korrektur mit Top- und Bottom-Coding und Einsatz von Mittelwert**

In [ ]:
#df = pd.DataFrame({'heart-rate': sample})
df['heart-rate_corrected_mean'] = df['heartrate'].apply(lambda x: mean_heart_rate if x < lower_bound_3 or x > upper_bound_3 else x)

# Plot
sample = df['heart-rate_corrected_mean'].copy()

if 'time' in df.columns:
    plt.figure(figsize=(8, 5))
    sns.lineplot(x=df['time'], y=sample, color='red')
    plt.xlabel('Zeit')
    plt.ylabel('Herzrate')
    plt.title('Liniendiagramm der Herzrate über die Zeit')
    plt.show()

**Korrektur durch Linare Interpolarisation**

In [ ]:
#df = pd.DataFrame({'heart-rate': sample})
df['heart-rate_corrected_interpolation'] = df['heartrate']
df.loc[(df['heart-rate_corrected_interpolation'] < lower_bound_3) | (df['heart-rate_corrected_interpolation'] > upper_bound_3), 'heart-rate_corrected_interpolation'] = np.nan

df['heart-rate_corrected_interpolation'] = df['heart-rate_corrected_interpolation'].interpolate()

# Plot
sample = df['heart-rate_corrected_interpolation'].copy()

if 'time' in df.columns:
    plt.figure(figsize=(8, 5))
    sns.lineplot(x=df['time'], y=sample, color='red')
    plt.xlabel('Zeit')
    plt.ylabel('Herzrate')
    plt.title('Liniendiagramm der Herzrate über die Zeit')
    plt.show()